In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
import requests
from langchain_community.tools import DuckDuckGoSearchRun

In [8]:
search_tool = DuckDuckGoSearchRun()

In [23]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

llm = ChatOpenAI(
    openai_api_key="api",
    openai_api_base="https://openrouter.ai/api/v1",
    model="google/gemini-2.5-flash"
)

In [24]:
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub

In [25]:
# Step 2 : Pull the ReAct prompt from Langchain hub
prompt = hub.pull('hwchase17/react')

In [26]:
@tool
def get_weather(city: str) -> str:

    """This is a function that gets the weather information of a given city"""
    api_key = 'api'
    base_url = 'http://api.openweathermap.org/data/2.5/weather'

    params = {
        'q': city,
        'appid': api_key,
        'units': 'metric'  # Use 'imperial' for Fahrenheit
    }

    response = requests.get(base_url, params=params)

    if response.status_code == 200:
        data = response.json()
        weather_details = {
            'city': data['name'],
            'temperature': data['main']['temp'],
            'weather': data['weather'][0]['description'],
            'humidity': data['main']['humidity'],
            'wind_speed': data['wind']['speed'],
        }
        return weather_details
    else:
        return {'error': 'City not found or API limit reached'}

# Example usage:
# print(get_weather('London'))

In [27]:
# Step 3: Create the React agent manually with the pulled prompt
agent = create_react_agent(
    llm = llm,
    tools = [search_tool, get_weather],
    prompt = prompt
)

In [28]:
# Calling the AgentExecutor
agent_executor = AgentExecutor(
    agent = agent,
    tools = [search_tool, get_weather],
    verbose=True
)

In [29]:
# Invoke the agent executor
response = agent_executor.invoke({"input":"Find the capital of Madhya Pradesh, and how is the weather in that city"})
print(response)



> Entering new AgentExecutor chain...
I need to find two pieces of information: first, the capital of Madhya Pradesh, and second, the current weather in that capital city. I will use duckduckgo_search for the first part and get_weather for the second.
Action: duckduckgo_search
Action Input: capital of Madhya PradeshIndore (/ ɪnˈdɔːr / ⓘ; ISO: Indaura, Hindi: [ɪn̪d̪ɔːr]) is the largest and most populous city in the Indian state of Madhya Pradesh . The commercial capital of the state, it has been declared as the cleanest city in India 8 years in a row. [15] It is also considered the largest education hub in central India and houses campuses of both the Indian Institute of Technology and the ... 1 day ago - Its capital is Bhopal . Other major cities includes Indore, Gwalior, Ujjain, Jabalpur and Sagar. Madhya Pradesh is the second largest Indian state by area and the fifth largest state by population with over 72 million residents. It borders the states of Rajasthan to the northwest, Ut

In [30]:
response['output']

'The capital of Madhya Pradesh is Bhopal, and the weather in that city is 21°C.'